# Notebook 26 — Regime-Shift Constraint Routing

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 25 evaluated finite constraint-budget allocation across route-memory policies.

Notebook 26 closes this adaptive-policy block by adding explicit nonstationary environments:

- stable routing,
- overload pressure,
- sparse-memory degradation,
- adversarial route transitions,
- recovery dynamics.

Constraint view:
> adaptive routing is useful only when policies remain coherent across changing regimes, not only during stable plateaus.


## Goals

1. Load Notebook 25 policy/budget outputs when available.
2. Construct a regime-shift environment over 240 windows.
3. Evaluate routing policies under stable, overload, sparse-memory, adversarial, and recovery phases.
4. Measure regime-conditioned constraint score, regret, recovery time, route persistence, and instability.
5. Export CSV, JSON, Markdown report, and PNG figures.
6. Generate a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension"
NOTEBOOKS_DIR = RML_ROOT / "notebooks"
RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 25 outputs

Preferred input:

```text
results/notebook25_constraint_budget_allocation.csv
```

If unavailable, this notebook creates a fallback budget-allocation stream.


In [ ]:
input_path = RESULTS_DIR / "notebook25_constraint_budget_allocation.csv"

policies = ["reactive", "predictive", "conservative_predictive", "aggressive_predictive", "cgcs_balanced"]
budget_policies = ["uniform_budget", "pressure_first", "forecast_first", "cgcs_balanced_budget", "regret_minimizing_budget"]
macro_routes = ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"]

if input_path.exists():
    base_df = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 25 output not found; creating fallback budget stream.")
    rng = np.random.default_rng(26)
    n = 240
    rows = []
    for t in range(n):
        macro = macro_routes[(t // 12 + (t // 55)) % len(macro_routes)]
        pressure = np.clip(0.45 + 0.22*np.sin(t/18) + rng.normal(0, 0.07), 0, 1)
        stability = np.clip(0.55 + 0.28*np.sin((t-92)/30) + rng.normal(0, 0.08), 0, 1)
        residual = np.clip(0.10 + 0.06*pressure + rng.normal(0, 0.025), 0, 1)
        forecast = np.clip(0.25 + 0.55*(pressure > 0.62) + rng.normal(0, 0.12), 0, 1)
        for p in policies:
            for b in budget_policies:
                bias = {
                    "reactive": -0.030,
                    "predictive": 0.020,
                    "conservative_predictive": 0.035,
                    "aggressive_predictive": 0.005,
                    "cgcs_balanced": 0.045,
                }[p]
                budget_bias = {
                    "uniform_budget": 0.005,
                    "pressure_first": 0.010,
                    "forecast_first": 0.015,
                    "cgcs_balanced_budget": 0.025,
                    "regret_minimizing_budget": 0.012,
                }[b]
                score = np.clip(0.55 + bias + budget_bias + 0.09*stability - 0.06*residual - 0.03*pressure + rng.normal(0, 0.012), 0, 1)
                regret = np.clip(0.07 + 0.05*pressure + 0.04*residual - 0.02*stability - bias + rng.normal(0, 0.01), 0, 1)
                rows.append({
                    "window": t,
                    "policy": p,
                    "budget_policy": b,
                    "macro_route": macro,
                    "pressure": pressure,
                    "rolling_pressure": pressure,
                    "rolling_stability": stability,
                    "rolling_residual": residual,
                    "forecast_probability": forecast,
                    "adjusted_constraint_score": score,
                    "budget_regret": regret,
                })
    base_df = pd.DataFrame(rows)

base_df.head()


## Build regime-shift environment

The environment is intentionally nonstationary. Each window receives a regime label and regime-specific perturbations.


In [ ]:
def assign_regime(t):
    if t < 48:
        return "stable"
    if t < 96:
        return "overload"
    if t < 144:
        return "sparse_memory"
    if t < 192:
        return "adversarial"
    return "recovery"

regime_order = ["stable", "overload", "sparse_memory", "adversarial", "recovery"]
regime_id = {r: i for i, r in enumerate(regime_order)}
regime_df = pd.DataFrame({"window": np.arange(240)})
regime_df["regime"] = regime_df["window"].map(assign_regime)
regime_df["regime_id"] = regime_df["regime"].map(regime_id)

regime_params = pd.DataFrame([
    {"regime": "stable", "pressure_shift": -0.08, "memory_decay": 0.05, "route_penalty": 0.02, "recovery_factor": 0.80},
    {"regime": "overload", "pressure_shift": 0.22, "memory_decay": 0.12, "route_penalty": 0.05, "recovery_factor": 0.55},
    {"regime": "sparse_memory", "pressure_shift": 0.10, "memory_decay": 0.30, "route_penalty": 0.08, "recovery_factor": 0.45},
    {"regime": "adversarial", "pressure_shift": 0.18, "memory_decay": 0.20, "route_penalty": 0.22, "recovery_factor": 0.35},
    {"regime": "recovery", "pressure_shift": -0.04, "memory_decay": 0.10, "route_penalty": 0.04, "recovery_factor": 0.75},
])

regime_df = regime_df.merge(regime_params, on="regime", how="left")
regime_df.head()


## Evaluate policies under regimes

Policy behavior is evaluated by regime-conditioned score, regret, volatility, and recovery response.


In [ ]:
rng = np.random.default_rng(260)

# Collapse Notebook 25 rows into one route-policy observation stream per window/policy.
if "budget_policy" in base_df.columns:
    preferred_budget = "cgcs_balanced_budget"
    if preferred_budget in set(base_df["budget_policy"]):
        stream = base_df[base_df["budget_policy"] == preferred_budget].copy()
    else:
        stream = base_df.copy()
else:
    stream = base_df.copy()

# Ensure only one row per window/policy.
num_cols = stream.select_dtypes(include=[np.number]).columns.tolist()
agg_cols = {c: "mean" for c in num_cols if c != "window"}
for c in ["macro_route"]:
    if c in stream.columns:
        agg_cols[c] = "first"
stream = stream.groupby(["window", "policy"], as_index=False).agg(agg_cols)

# Fill missing rows if loaded source is sparse.
full_index = pd.MultiIndex.from_product([np.arange(240), policies], names=["window", "policy"]).to_frame(index=False)
stream = full_index.merge(stream, on=["window", "policy"], how="left")
for c in ["rolling_pressure", "pressure"]:
    if c not in stream.columns:
        stream[c] = np.nan
stream["pressure_base"] = stream["rolling_pressure"].fillna(stream["pressure"]).fillna(0.5)
stream["stability_base"] = stream.get("rolling_stability", pd.Series(np.nan, index=stream.index)).fillna(0.55)
stream["residual_base"] = stream.get("rolling_residual", pd.Series(np.nan, index=stream.index)).fillna(0.10)
stream["forecast_base"] = stream.get("forecast_probability", pd.Series(np.nan, index=stream.index)).fillna(0.30)
stream["score_base"] = stream.get("adjusted_constraint_score", pd.Series(np.nan, index=stream.index)).fillna(0.58)
stream["regret_base"] = stream.get("budget_regret", pd.Series(np.nan, index=stream.index)).fillna(0.09)
stream["macro_route"] = stream.get("macro_route", pd.Series(index=stream.index, dtype=object)).fillna(
    stream["window"].map(lambda x: macro_routes[(x // 16) % len(macro_routes)])
)

model_df = stream.merge(regime_df, on="window", how="left")

policy_traits = {
    "reactive": {"adapt": 0.25, "risk": 0.10, "recovery": 0.30, "overload_bonus": -0.04, "adversarial_penalty": 0.08},
    "predictive": {"adapt": 0.60, "risk": 0.22, "recovery": 0.55, "overload_bonus": 0.04, "adversarial_penalty": 0.07},
    "conservative_predictive": {"adapt": 0.50, "risk": 0.08, "recovery": 0.65, "overload_bonus": 0.01, "adversarial_penalty": 0.02},
    "aggressive_predictive": {"adapt": 0.82, "risk": 0.36, "recovery": 0.40, "overload_bonus": 0.08, "adversarial_penalty": 0.13},
    "cgcs_balanced": {"adapt": 0.68, "risk": 0.12, "recovery": 0.78, "overload_bonus": 0.04, "adversarial_penalty": 0.03},
}

rows = []
for _, row in model_df.iterrows():
    p = row["policy"]
    tr = policy_traits[p]
    regime = row["regime"]
    pressure = np.clip(row["pressure_base"] + row["pressure_shift"] + rng.normal(0, 0.025), 0, 1)
    memory_quality = np.clip(row["stability_base"] - row["memory_decay"] + 0.10*tr["recovery"] + rng.normal(0, 0.025), 0, 1)
    route_volatility = np.clip(0.22 + 0.55*pressure + row["route_penalty"] + tr["risk"] - 0.25*tr["adapt"] + rng.normal(0, 0.03), 0, 1)
    decompression_need = np.clip(0.15 + 0.55*pressure + 0.40*row["memory_decay"] + 0.20*row["route_penalty"] - 0.20*tr["recovery"] + rng.normal(0, 0.03), 0, 1)
    regime_bonus = 0
    if regime == "overload":
        regime_bonus += tr["overload_bonus"]
    if regime == "adversarial":
        regime_bonus -= tr["adversarial_penalty"]
    if regime == "recovery":
        regime_bonus += 0.06 * tr["recovery"]
    if regime == "sparse_memory":
        regime_bonus += 0.04 * tr["adapt"] - 0.10 * row["memory_decay"]

    constraint_score = np.clip(
        row["score_base"] + regime_bonus + 0.16*memory_quality - 0.10*pressure - 0.10*route_volatility - 0.07*decompression_need + rng.normal(0, 0.012),
        0, 1,
    )
    regret = np.clip(
        row["regret_base"] + 0.20*route_volatility + 0.12*decompression_need + 0.10*row["route_penalty"] - 0.10*tr["adapt"] + rng.normal(0, 0.008),
        0, 1,
    )
    route_persistence = np.clip(1 - route_volatility + 0.15*memory_quality, 0, 1)
    recovery_score = np.clip(row["recovery_factor"] * tr["recovery"] + 0.35*memory_quality - 0.20*decompression_need, 0, 1)
    recommendation_score = np.clip(0.38*constraint_score + 0.22*recovery_score + 0.20*route_persistence - 0.20*regret, 0, 1)

    rows.append({
        "window": int(row["window"]),
        "regime": regime,
        "regime_id": int(row["regime_id"]),
        "policy": p,
        "macro_route": row["macro_route"],
        "pressure": pressure,
        "memory_quality": memory_quality,
        "route_volatility": route_volatility,
        "decompression_need": decompression_need,
        "constraint_score": constraint_score,
        "regret": regret,
        "route_persistence": route_persistence,
        "recovery_score": recovery_score,
        "recommendation_score": recommendation_score,
    })

eval_df = pd.DataFrame(rows)
eval_df.head()


## Summaries and transition matrices

In [ ]:
summary = pd.DataFrame([{
    "windows": eval_df["window"].nunique(),
    "regimes": eval_df["regime"].nunique(),
    "policies": eval_df["policy"].nunique(),
    "mean_constraint_score": eval_df["constraint_score"].mean(),
    "mean_regret": eval_df["regret"].mean(),
    "mean_recovery_score": eval_df["recovery_score"].mean(),
    "mean_route_persistence": eval_df["route_persistence"].mean(),
    "mean_recommendation_score": eval_df["recommendation_score"].mean(),
}])

policy_summary = eval_df.groupby("policy").agg(
    windows=("window", "nunique"),
    mean_constraint_score=("constraint_score", "mean"),
    mean_regret=("regret", "mean"),
    mean_recovery_score=("recovery_score", "mean"),
    mean_route_persistence=("route_persistence", "mean"),
    mean_route_volatility=("route_volatility", "mean"),
    mean_memory_quality=("memory_quality", "mean"),
    mean_recommendation_score=("recommendation_score", "mean"),
).reset_index().sort_values("mean_recommendation_score", ascending=False)

regime_summary = eval_df.groupby(["regime", "policy"]).agg(
    windows=("window", "nunique"),
    mean_constraint_score=("constraint_score", "mean"),
    mean_regret=("regret", "mean"),
    mean_recovery_score=("recovery_score", "mean"),
    mean_route_persistence=("route_persistence", "mean"),
    mean_route_volatility=("route_volatility", "mean"),
    mean_decompression_need=("decompression_need", "mean"),
    mean_recommendation_score=("recommendation_score", "mean"),
).reset_index()

# Regime transition matrix over windows.
window_regimes = regime_df.sort_values("window")["regime"].to_numpy()
transition_counts = pd.DataFrame(0, index=regime_order, columns=regime_order, dtype=float)
for a, b in zip(window_regimes[:-1], window_regimes[1:]):
    transition_counts.loc[a, b] += 1
transition_matrix = transition_counts.div(transition_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

# Route persistence lengths for best policy per window.
best_by_window = eval_df.sort_values("recommendation_score", ascending=False).groupby("window").head(1).sort_values("window")
segments = []
start = int(best_by_window.iloc[0]["window"])
last_route = best_by_window.iloc[0]["macro_route"]
last_policy = best_by_window.iloc[0]["policy"]
prev = start
for _, r in best_by_window.iloc[1:].iterrows():
    cur = int(r["window"])
    if r["macro_route"] != last_route or r["policy"] != last_policy:
        segments.append({"start": start, "end": prev, "length": prev-start+1, "macro_route": last_route, "policy": last_policy})
        start = cur
        last_route = r["macro_route"]
        last_policy = r["policy"]
    prev = cur
segments.append({"start": start, "end": prev, "length": prev-start+1, "macro_route": last_route, "policy": last_policy})
route_persistence_df = pd.DataFrame(segments)

summary


## Save CSV and JSON outputs

In [ ]:
eval_csv = RESULTS_DIR / "notebook26_regime_shift_constraint_routing.csv"
summary_csv = RESULTS_DIR / "notebook26_policy_summary.csv"
regime_csv = RESULTS_DIR / "notebook26_regime_summary.csv"
transition_csv = RESULTS_DIR / "notebook26_regime_transition_matrix.csv"
persistence_csv = RESULTS_DIR / "notebook26_route_persistence.csv"
json_path = RESULTS_DIR / "notebook26_regime_shift_constraint_routing.json"

eval_df.to_csv(eval_csv, index=False)
policy_summary.to_csv(summary_csv, index=False)
regime_summary.to_csv(regime_csv, index=False)
transition_matrix.to_csv(transition_csv)
route_persistence_df.to_csv(persistence_csv, index=False)

payload = {
    "summary": summary.to_dict(orient="records"),
    "policy_summary": policy_summary.to_dict(orient="records"),
    "regime_summary": regime_summary.to_dict(orient="records"),
    "regime_transition_matrix": transition_matrix.reset_index(names="current_regime").to_dict(orient="records"),
}
json_path.write_text(json.dumps(payload, indent=2))

print(eval_csv)
print(summary_csv)
print(regime_csv)
print(transition_csv)
print(persistence_csv)
print(json_path)


## Figures

In [ ]:

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(path)

# 1. Regime timeline
plt.figure(figsize=(14, 4))
plt.step(regime_df["window"], regime_df["regime_id"], where="post")
plt.yticks(list(regime_id.values()), list(regime_id.keys()))
plt.xlabel("Window")
plt.ylabel("Regime")
plt.title("Regime-Shift Constraint Routing: Regime Timeline")
savefig("notebook26_regime_timeline.png")

# 2. Mean constraint score by regime and policy
pivot = regime_summary.pivot(index="regime", columns="policy", values="mean_constraint_score").reindex(regime_order)
plt.figure(figsize=(12, 6))
plt.imshow(pivot.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=40, ha="right")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label="Mean constraint score")
plt.title("Regime-Shift Constraint Routing: Constraint Score by Regime")
savefig("notebook26_constraint_score_by_regime.png")

# 3. Regret by regime
pivot_regret = regime_summary.pivot(index="regime", columns="policy", values="mean_regret").reindex(regime_order)
plt.figure(figsize=(12, 6))
plt.imshow(pivot_regret.values, aspect="auto", vmin=0, vmax=max(0.35, float(pivot_regret.max().max())))
plt.xticks(range(len(pivot_regret.columns)), pivot_regret.columns, rotation=40, ha="right")
plt.yticks(range(len(pivot_regret.index)), pivot_regret.index)
plt.colorbar(label="Mean regret")
plt.title("Regime-Shift Constraint Routing: Regret by Regime")
savefig("notebook26_regret_by_regime.png")

# 4. Recovery curve
plt.figure(figsize=(14, 5))
for p in policies:
    s = eval_df[eval_df["policy"] == p].groupby("window")["recovery_score"].mean()
    plt.plot(s.index, s.values, label=p)
plt.xlabel("Window")
plt.ylabel("Recovery score")
plt.title("Regime-Shift Constraint Routing: Recovery Curve")
plt.legend()
savefig("notebook26_recovery_curve.png")

# 5. Policy frontier
plt.figure(figsize=(8, 6))
frontier = policy_summary.copy()
plt.scatter(frontier["mean_regret"], frontier["mean_constraint_score"], s=120)
for _, r in frontier.iterrows():
    plt.text(r["mean_regret"], r["mean_constraint_score"], r["policy"])
plt.xlabel("Mean regret (lower is better)")
plt.ylabel("Mean constraint score (higher is better)")
plt.title("Regime-Shift Constraint Routing: Regret / Constraint Frontier")
savefig("notebook26_policy_frontier.png")

# 6. Regime transition matrix
plt.figure(figsize=(8, 6))
plt.imshow(transition_matrix.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(transition_matrix.columns)), transition_matrix.columns, rotation=40, ha="right")
plt.yticks(range(len(transition_matrix.index)), transition_matrix.index)
plt.xlabel("Next regime")
plt.ylabel("Current regime")
plt.colorbar(label="Transition probability")
plt.title("Regime-Shift Constraint Routing: Regime Transition Matrix")
savefig("notebook26_regime_transition_matrix.png")

# 7. Memory quality timeline
plt.figure(figsize=(14, 5))
for p in policies:
    s = eval_df[eval_df["policy"] == p].groupby("window")["memory_quality"].mean().rolling(5, min_periods=1).mean()
    plt.plot(s.index, s.values, label=p)
plt.xlabel("Window")
plt.ylabel("Memory quality")
plt.title("Regime-Shift Constraint Routing: Memory Quality Timeline")
plt.legend()
savefig("notebook26_memory_quality_timeline.png")

# 8. Route persistence distribution
plt.figure(figsize=(10, 5))
route_persistence_df.groupby("policy")["length"].mean().sort_values().plot(kind="bar")
plt.xlabel("Policy")
plt.ylabel("Mean route segment length")
plt.title("Regime-Shift Constraint Routing: Route Persistence")
savefig("notebook26_route_persistence.png")

# 9. Recommendation score
plt.figure(figsize=(12, 5))
policy_summary.set_index("policy")["mean_recommendation_score"].sort_values().plot(kind="bar")
plt.xlabel("Policy")
plt.ylabel("Recommendation score")
plt.title("Regime-Shift Constraint Routing: Adaptive Policy Recommendation Score")
savefig("notebook26_recommendation_score.png")

# 10. Volatility timeline
plt.figure(figsize=(14, 5))
for p in policies:
    s = eval_df[eval_df["policy"] == p].groupby("window")["route_volatility"].mean().rolling(5, min_periods=1).mean()
    plt.plot(s.index, s.values, label=p)
plt.xlabel("Window")
plt.ylabel("Route volatility")
plt.title("Regime-Shift Constraint Routing: Route Volatility Timeline")
plt.legend()
savefig("notebook26_route_volatility_timeline.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
def md_table(df, max_rows=20):
    return df.head(max_rows).to_markdown(index=False)

figure_names = [
    "notebook26_regime_timeline.png",
    "notebook26_constraint_score_by_regime.png",
    "notebook26_regret_by_regime.png",
    "notebook26_recovery_curve.png",
    "notebook26_policy_frontier.png",
    "notebook26_regime_transition_matrix.png",
    "notebook26_memory_quality_timeline.png",
    "notebook26_route_persistence.png",
    "notebook26_recommendation_score.png",
    "notebook26_route_volatility_timeline.png",
]

report_path = REPORTS_DIR / "report_26_regime_shift_constraint_routing.md"

output_links = "\n".join([
    '- Regime-shift routing CSV: <a href="results/notebook26_regime_shift_constraint_routing.csv">`results/notebook26_regime_shift_constraint_routing.csv`</a>',
    '- Regime-shift routing JSON: <a href="results/notebook26_regime_shift_constraint_routing.json">`results/notebook26_regime_shift_constraint_routing.json`</a>',
    '- Policy summary CSV: <a href="results/notebook26_policy_summary.csv">`results/notebook26_policy_summary.csv`</a>',
    '- Regime summary CSV: <a href="results/notebook26_regime_summary.csv">`results/notebook26_regime_summary.csv`</a>',
    '- Regime transition matrix CSV: <a href="results/notebook26_regime_transition_matrix.csv">`results/notebook26_regime_transition_matrix.csv`</a>',
    '- Route persistence CSV: <a href="results/notebook26_route_persistence.csv">`results/notebook26_route_persistence.csv`</a>',
] + [f'- Figure: <a href="figures/{name}">`figures/{name}`</a>' for name in figure_names])

report = f"""# Report 26 — Regime-Shift Constraint Routing

Notebook 26 evaluates route-memory policies under nonstationary operating regimes.

Constraint view:
> adaptive routing is useful only when policies remain coherent across changing regimes, not only during stable plateaus.

## Generated outputs

{output_links}

## Summary

{summary.to_markdown(index=False)}

## Policy summary

{policy_summary.to_markdown(index=False)}

## Regime summary

{regime_summary.to_markdown(index=False)}

## Regime transition probabilities

{transition_matrix.to_markdown()}

## Interpretation

- Stable regimes reward route persistence and low decompression pressure.
- Overload regimes reward predictive and pressure-aware adaptation.
- Sparse-memory regimes expose whether a policy can recover from degraded retrieval quality.
- Adversarial regimes penalize aggressive rerouting and highlight robustness differences.
- Recovery regimes test whether policies can return to coherent compressed routing after instability.

## Next step

Notebook 27 can start a new group: distributed adaptive coordination across multiple routing agents.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])

## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook26_regime_shift_constraint_routing_outputs.zip"
export_path = RML_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook26_*"):
        zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
    for p in FIGURES_DIR.glob("notebook26_*"):
        zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
    for p in REPORTS_DIR.glob("report_26_*"):
        zf.write(p, arcname=str(p.relative_to(RML_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook26_regime_shift_constraint_routing_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook26_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_26_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
